# Сравнение частоты створов

Источник: Dzammbo/Scaner, архив анализа 39743379318-1, commit 16afe0889d3835bc157aa3cdb74e0fe0ff47ac64. Проверяется тот же зафиксированный набор, что в исходном разборе. Код построения всех сравнений: `analysis.py`; полный результат: `sot-comparison.json`.

Низкая частота: <0,1; средняя: от 0,1 до <0,2; высокая: от 0,2 удара в створ обеих команд на минуту. Показатель накопленный. Пропуски отделены от нулей.

In [1]:
from pathlib import Path
import json, gzip, math
from decimal import Decimal as Q
base = Path.cwd()
source = base / "entry-points.json.gz"
if not source.exists():
    source = next(p / "research/all-entry-audit/2026-10-22/39743379318-1/entry-points.json.gz" for p in [base, *base.parents] if (p / "research/all-entry-audit/2026-10-22/39743379318-1/entry-points.json.gz").exists())
rows = json.loads(gzip.decompress(source.read_bytes()))
rows = [r for r in rows if r["basis"] == "exact_provider_main" and r["profit"] is not None and r["minute"] > 0]
for r in rows:
    line, price, total = Q(str(r["line"])), Q(str(r["price"])), Q(sum(r["final"]))
    half = (line * 2).to_integral_value(rounding="ROUND_FLOOR") / 2
    legs = [line] if line * 2 == (line * 2).to_integral_value() else [half, half + Q(".5")]
    profit = sum(price - 1 if total > x else Q(0) if total == x else Q(-1) for x in legs) / len(legs)
    assert abs(float(profit) - r["profit"]) < 1e-9
low = {}
first_need2 = {}
for r in sorted(rows, key=lambda r: (r["t"], r["order"])):
    if r["need"] != 2: continue
    first_need2.setdefault(r["id"], r)
    if r["sot"] is not None and r["sot"] / r["minute"] < .1:
        low.setdefault(r["id"], r)
assert len(low) == 573
roi = sum(r["profit"] for r in low.values()) / len(low)
assert abs(roi - .05015270506108202) < 1e-12
print("Независимо проверено расчётов:", len(rows))
print("Исходный карман:", len(low), "матчей; ROI", round(roi * 100, 2), "%")
print("Первая точка need=2 до деления по створам:", len(first_need2), "матчей")


Независимо проверено расчётов: 14141
Исходный карман: 573 матчей; ROI 5.02 %
Первая точка need=2 до деления по створам: 899 матчей


## Метод

Для оценки исходного кармана берётся первый подходящий вход на матч. Для сравнения частоты сначала выбирается первый вход с необходимыми двумя голами независимо от числа створов, затем определяется группа. Поэтому группы сравнения не пересекаются. Сопоставление без возвращения: одинаковые счёт хозяев и гостей и линия, коэффициент отличается не более чем на 0,05, минута не более чем на 3; отдельно проверяется точное совпадение минуты, один регион, один календарный день и одна лига. Выбор пар не использует исходы. Интервалы неопределённости получены повторной выборкой пар и не учитывают исходный перебор гипотез.

Календарная проверка: 21.10.2026 10:21:53 - 22.10.2026 10:21:53 МСК. Это ретроспективная проверка на выделенном отрезке, не новый независимый прогнозный эксперимент.

In [2]:
report = json.loads((base / "sot-comparison.json").read_text())
comparisons = report["cohorts"]["need2"]["matched"]["all"]
for name in ["minute3_price005", "exact_minute_price005", "minute3_price005_same_region", "minute3_price005_same_day"]:
    m = comparisons[name]
    seen = set()
    for left, right in m["ids"]:
        a, b = first_need2[left], first_need2[right]
        assert left not in seen and right not in seen
        seen.update([left, right])
        assert a["score"] == b["score"] and a["line"] == b["line"]
        assert a["sot"] / a["minute"] < .1 <= b["sot"] / b["minute"]
        assert abs(a["price"] - b["price"]) <= .05 + 1e-9
        assert abs(a["minute"] - b["minute"]) <= (0 if name == "exact_minute_price005" else 3)
    left_roi = sum(first_need2[a]["profit"] for a, b in m["ids"]) / m["pairs"]
    right_roi = sum(first_need2[b]["profit"] for a, b in m["ids"]) / m["pairs"]
    assert abs(left_roi - m["low"]["roi"]) < 1e-12
    assert abs(right_roi - m["control"]["roi"]) < 1e-12
    print(name, "пар", m["pairs"], "ROI:", round(left_roi * 100, 2), "/", round(right_roi * 100, 2))


minute3_price005 пар 210 ROI: 5.26 / -7.65
exact_minute_price005 пар 119 ROI: 1.43 / 7.65
minute3_price005_same_region пар 148 ROI: -3.26 / -4.76
minute3_price005_same_day пар 151 ROI: 0.41 / 0.66


## Вывод

Исходный плюс сохраняется после исключения нулевых створов. Однако преимущество низкой частоты перед более высокой неустойчиво к условиям сравнения. При точном совпадении минуты положительного преимущества не найдено, а интервалы неопределённости широки. Результат не доказывает, что низкая частота створов сама по себе увеличивает вероятность будущих голов. Пропуски кодом не преобразуются в нули; качество всех статистических полей самого поставщика независимо не подтверждено.